##### **Norwegian Mobile Telecommunications — Commercial Performance & Competitive Analysis (2015–2025)**


###### The goal of the analysis is to analyze and understand:

1. Who is winning in the Norwegian telecom market?
2. What is driving their performance?
3. Where are the commercial risks?



###### **Commercial questions to investigate and I would love to answer**

1. Which operators generated the most revenue between 2015 and 2025?

2. What percentage of the market's cumulative revenue each operator generated.

3. How is the revenue growing?

4. What is the subscription volumes by operator?

5. How is the subscription growing?

6. Is growth stronger in consumer or business markets?

7. Which mobile operators are most dependent on the slower-growing business segment?



###### **English Translation of some of the Norwegian columns/ words for my audience and context**.

1. Hovedkategori: Service categories
2. Hovedgruppe: Measurement types
3. Markedssegment: Market segments
4. Delkategori: Service subcategories
5. Teknologi: Technologies
6. Rapport: Reporting Period




**Francis Ojinnaka**


In [1]:
## Importing relevant libraries for my analysis

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

plt.style.use('bmh') #style format to use



StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 3, Finished, Available, Finished, False)

In [2]:
## Loading the Ekomstat data in the OneLake and performing an exploratory analysis
#  Selecting the columns needed for analysis 


path = 'abfss://Francis_WK@onelake.dfs.fabric.microsoft.com/Norway_Telco.Lakehouse/Files/Nkom_files/Ekomstat_Alle-data.xlsx'


datatypes = {'Hovedgruppe_id': 'int8', 'Tilbyder_id': 'int16', 'svar': 'int32'}

cols = ["Hovedkategori", 'Tilbyder_id', 'Hovedgruppe_id', 'Delkategori','Teknologi','NivÃ¥1',
         'Markedssegment', 'Rapport', 'svar' ]




#loading the fact table

## Preparing the table for my analysis and filtering on the measurement types I am interested in
## Since I am working on Mobile Telephone communications, i want to look at the Revenue and subscription related KPIs
# 1 == abonnement (subscription) and 2 == innteker (revenue)

ekomstat = pd.read_excel(path, usecols= cols, sheet_name=0, dtype= datatypes )

ekomstat = ekomstat.query(
     "Hovedgruppe_id  in (1, 2) and Delkategori == 'Mobiltelefoni'"

).reset_index()

ekomstat

#loading dimension tables
dtypes = {'Hovedgruppe_id': 'int8'}
hoved_ids = pd.read_excel(path,  sheet_name=1, dtype= dtypes, nrows= 2)

dtyp = {'Tilbyder_id': 'int16'}
tilbyder_ids = pd.read_excel(path,  sheet_name=2, dtype= dtyp)


## Data and table statistics to understand the data I am working on
ekomstat.head(10)
ekomstat.info()
ekomstat.describe(include= 'all').round()
ekomstat.dtypes
ekomstat.shape

hoved_ids.head()
hoved_ids.head()

tilbyder_ids.dtypes, hoved_ids.dtypes

tilbyder_ids['Tilbyder'] = tilbyder_ids['Tilbyder'].str.title()

tilbyder_ids
hoved_ids

ekomstat['NivÃ¥1'] = ekomstat['NivÃ¥1'].str.strip()
ekomstat['Markedssegment'] = ekomstat['Markedssegment'].str.strip()

ekomstat

StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 4, Finished, Available, Finished, False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3904 entries, 0 to 3903
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   index           3904 non-null   int64 
 1   Hovedkategori   3904 non-null   object
 2   Tilbyder_id     3904 non-null   int16 
 3   Hovedgruppe_id  3904 non-null   int8  
 4   Delkategori     3904 non-null   object
 5   Teknologi       3904 non-null   object
 6   NivÃ¥1          3904 non-null   object
 7   Markedssegment  3904 non-null   object
 8   Rapport         3904 non-null   object
 9   svar            3904 non-null   int32 
dtypes: int16(1), int32(1), int64(1), int8(1), object(6)
memory usage: 240.3+ KB


,index,Hovedkategori,Tilbyder_id,Hovedgruppe_id,Delkategori,Teknologi,NivÃ¥1,Markedssegment,Rapport,svar
0,2769,Mobiltjenester,12,1,Mobiltelefoni,Ingen,Fakturert,Bedrift,2017-HelÃ¥r,760
1,2770,Mobiltjenester,12,1,Mobiltelefoni,Ingen,Fakturert,Bedrift,2018-HalvÃ¥r,4467
2,2771,Mobiltjenester,12,1,Mobiltelefoni,Ingen,Fakturert,Bedrift,2018-HelÃ¥r,9443
3,2772,Mobiltjenester,12,1,Mobiltelefoni,Ingen,Fakturert,Bedrift,2019-HalvÃ¥r,11869
4,2773,Mobiltjenester,12,1,Mobiltelefoni,Ingen,Fakturert,Bedrift,2019-HelÃ¥r,14069
...,...,...,...,...,...,...,...,...,...,...
3899,62145,Mobiltjenester,170,2,Mobiltelefoni,Ingen,Trafikkfakturert,Privat,2023-HalvÃ¥r,87
3900,62146,Mobiltjenester,170,2,Mobiltelefoni,Ingen,Trafikkfakturert,Privat,2023-HelÃ¥r,184
3901,62147,Mobiltjenester,170,2,Mobiltelefoni,Ingen,Trafikkfakturert,Privat,2024-HalvÃ¥r,94
3902,62148,Mobiltjenester,170,2,Mobiltelefoni,Ingen,Trafikkfakturert,Privat,2024-HelÃ¥r,197


In [3]:
## Merging tables to get the lookup columns for business context using an inner join


ekomstat_1 = ekomstat.merge(hoved_ids[['Hovedgruppe', 'Hovedgruppe_id']],
                        how = 'inner',
                        left_on = 'Hovedgruppe_id',
                        right_on= 'Hovedgruppe_id'
)

ekomstat_1.info()
ekomstat_1.head()

ekom_2 = ekomstat_1.merge(
    tilbyder_ids,
    on = 'Tilbyder_id',
    how = 'inner'
)

ekom_2.head()
ekom_2.info()

## Dropping redundant columns

ekom_new = ekom_2.drop(['Tilbyder_id','Hovedgruppe_id' ], axis = 1)

ekom_new.head(30)




ekom_new['Hovedgruppe'].value_counts()
ekom_new['Tilbyder'].value_counts()

ekom_new.head()


## Data manipulations and transformations
#1. Extracting the Year Part from rapport
#2. Extracting the full and the half year reporting period
#3. Filtering on the Full year rows for the analysis

ekom_new = ekom_new.assign(
    Year = ekom_new['Rapport'].str[:4].astype('int16'),
    Half_FullYear = np.where(ekom_new['Rapport'].str.contains('Hel'), "Full Year", 'Half Year' )
    
)

ekom_new.head()

ekom_11 = ekom_new.drop(columns= ['Rapport', 'Hovedkategori'])


ekom_11 = ekom_11.query(
    "Half_FullYear == 'Full Year' "
)
ekom_11.reset_index(drop = True)

StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 5, Finished, Available, Finished, False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3904 entries, 0 to 3903
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   index           3904 non-null   int64 
 1   Hovedkategori   3904 non-null   object
 2   Tilbyder_id     3904 non-null   int16 
 3   Hovedgruppe_id  3904 non-null   int8  
 4   Delkategori     3904 non-null   object
 5   Teknologi       3904 non-null   object
 6   NivÃ¥1          3904 non-null   object
 7   Markedssegment  3904 non-null   object
 8   Rapport         3904 non-null   object
 9   svar            3904 non-null   int32 
 10  Hovedgruppe     3904 non-null   object
dtypes: int16(1), int32(1), int64(1), int8(1), object(7)
memory usage: 270.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3904 entries, 0 to 3903
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   index           3904 non-null   int64 
 

,index,Delkategori,Teknologi,NivÃ¥1,Markedssegment,svar,Hovedgruppe,Tilbyder,Year,Half_FullYear
0,2769,Mobiltelefoni,Ingen,Fakturert,Bedrift,760,Abonnement,Atea,2017,Full Year
1,2771,Mobiltelefoni,Ingen,Fakturert,Bedrift,9443,Abonnement,Atea,2018,Full Year
2,2773,Mobiltelefoni,Ingen,Fakturert,Bedrift,14069,Abonnement,Atea,2019,Full Year
3,2775,Mobiltelefoni,Ingen,Fakturert,Bedrift,18004,Abonnement,Atea,2020,Full Year
4,2777,Mobiltelefoni,Ingen,Fakturert,Bedrift,18040,Abonnement,Atea,2021,Full Year
...,...,...,...,...,...,...,...,...,...,...
1949,62140,Mobiltelefoni,Ingen,Trafikkfakturert,Privat,108,Inntekter,Xplora Mobile,2020,Full Year
1950,62142,Mobiltelefoni,Ingen,Trafikkfakturert,Privat,138,Inntekter,Xplora Mobile,2021,Full Year
1951,62144,Mobiltelefoni,Ingen,Trafikkfakturert,Privat,162,Inntekter,Xplora Mobile,2022,Full Year
1952,62146,Mobiltelefoni,Ingen,Trafikkfakturert,Privat,184,Inntekter,Xplora Mobile,2023,Full Year


In [20]:

#Q.1 Which operators generated the most revenue between 2015 and 2025?


reve_ = ekom_11.query(
    " Hovedgruppe == 'Inntekter' "
).groupby(
   
        ['Year','Tilbyder', 'Markedssegment'], as_index= 0
).agg(
    Total_Revenue = ('svar', 'sum')
).sort_values(['Total_Revenue'], ascending = False )

reve_ =reve_.assign( Pct_of_Total=  round(reve_['Total_Revenue']/reve_.groupby(['Year'])['Total_Revenue'].transform('sum') * 100,1),
                     Operators_category = np.where(reve_['Tilbyder'] == 'Telenor Norge', 'Telenor Norge',np.where(reve_['Tilbyder'] == 'Telia Norge', 'Telia Norge',np.where(
                      reve_['Tilbyder'] == 'Lyse Tele', 'Lyse Tele' , np.where( reve_['Tilbyder'] == 'Ice Norge', 'Ice Norge', np.where(
                     reve_['Tilbyder'] == 'Unifon', 'Unifon', np.where(reve_['Tilbyder'] == 'Fjordkraft Mobil', 'Fjordkraft Mobil', 'Others')) ))))
                   
                     )

reve_.reset_index(drop = True)

reve_.head()

## By abolute values


pivot_revs = reve_.pivot_table(
    index = 'Tilbyder',
    columns = ['Year'],
    values = 'Total_Revenue',
    aggfunc = 'sum',
    fill_value = 0,
    margins ='All',
    margins_name = 'Total_Revenue'
).sort_values(['Total_Revenue'], ascending = 0)

pivot_revs = pivot_revs.iloc[1:]#.style.background_gradient(cmap = 'RdYlGn', axis =1,low=0.75, high=1.0) #across columns, axis = 0 (rows)
pivot_revs


## By pct of total for each year
pivot_pct = reve_.pivot_table(
    index='Tilbyder',
    columns='Year',
    values='Pct_of_Total',
    aggfunc='sum',
    fill_value=0
)

pivot_pct = pivot_pct.sort_values(
    by=pivot_pct.columns[-1],
    ascending=False
)

pivot_revs




reve_plot = (
    reve_.groupby(['Year', 'Operators_category'], as_index=False)['Total_Revenue'].sum()
)
reve_plot['Year'] = reve_plot['Year'].astype(str)  

fig = px.bar(
    reve_plot,
    x='Year',
    y='Total_Revenue',
    color='Operators_category',
    labels= {'Total_Revenue': 'Revenue'},
    title='Total Revenue from 2015-2025 by Norway Telcos',
    color_discrete_sequence=px.colors.qualitative.Bold,  
    
)
fig.update_xaxes(type='category')
fig.show()

pivot_pct.head(6)




StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 77, Finished, Available, Finished, False)

Year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Tilbyder,,,,,,,,,,,
Telenor Norge,59.3,58.3,58.3,56.7,56.3,56.3,54.9,53.5,51.3,50.9,50.7
Telia Norge,36.9,37.0,35.3,34.3,33.5,31.5,31.5,31.5,32.5,31.1,30.4
Lyse Tele,0.0,0.0,0.0,0.0,0.0,7.5,8.4,8.7,9.8,11.1,12.1
Unifon,0.0,0.0,0.0,0.2,0.6,1.0,1.3,1.9,2.3,2.4,2.3
Fjordkraft Mobil,0.1,0.1,0.3,0.8,1.1,1.6,1.5,1.5,1.3,1.3,1.2
Chili Mobil,0.2,0.2,0.2,0.5,1.0,1.1,1.1,1.1,1.2,1.2,1.1


In [5]:
display(reve_)

StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 31e004d9-f5d6-4aa9-9b42-16725ba39277)

In [6]:

#Q2 what percentage of the market's cumulative revenue each operator generated.

tilby_rev= ekom_11.query(
    " Hovedgruppe == 'Inntekter' "
).groupby(
   
        ['Tilbyder'], as_index= 0
).agg(
    Cummulative_Revenue = ('svar', 'sum')
).sort_values(['Cummulative_Revenue'], ascending = False )



tilby_rev = tilby_rev.assign( Pct_of_Cumm_Total =  lambda cumtotal: (cumtotal['Cummulative_Revenue']/cumtotal['Cummulative_Revenue'].sum())*100).round(1)

tilby_rev.reset_index(drop = True)



cum_charts = tilby_rev.head(6)

fig = px.bar(cum_charts, x= 'Tilbyder', y='Cummulative_Revenue',
 color='Tilbyder',  text_auto=True,    labels={'Cummulative_Revenue':'Revenue', 'Tilbyder': 'Operators'},
title = 'Top 6 Cummulative Revenue by Telco. Operators')
fig.show()


tilby_rev.head(10)


StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 8, Finished, Available, Finished, False)

,Tilbyder,Cummulative_Revenue,Pct_of_Cumm_Total
24,Telenor Norge,111011112,54.9
25,Telia Norge,66836120,33.1
14,Lyse Tele,11191834,5.5
8,Ice Norge,3317272,1.6
26,Unifon,2363906,1.2
4,Fjordkraft Mobil,2066390,1.0
3,Chili Mobil,1655503,0.8
15,Nextgentel,634939,0.3
12,Lycamobile Norway Ltd,555135,0.3
6,Happybytes,514269,0.3


In [7]:
#Q3 How is the revenue growing?


reve_growth = ekom_11.query(
    " Hovedgruppe == 'Inntekter' "
).groupby(
   
        ['Year'], as_index= 0
).agg(
    Revenue_growth = ('svar', 'sum')

).sort_values(['Year'], ascending = True )





Yearly_share = ekom_11.query(
    " Hovedgruppe == 'Inntekter' "
).groupby(
   
        ['Year'], as_index= 0
).agg(
    Total_Revenue = ('svar', 'sum')

 ).sort_values(['Total_Revenue'], ascending = False )

Yearly_share = Yearly_share.sort_values([ 'Year'])

Yearly_share['Previous_Year_Revenue'] = (
     Yearly_share['Total_Revenue'].shift(1)
 )

Yearly_share = Yearly_share.assign(
    Revenue_vs_LY = Yearly_share['Total_Revenue'] - Yearly_share['Previous_Year_Revenue'],
 MKT_growth = round(((Yearly_share['Total_Revenue']/Yearly_share['Previous_Year_Revenue']) - 1) * 100, 1)
)
display(Yearly_share)


# fig = px.bar(Yearly_share, x= 'Year', y='Cummulative_Revenue', color='Tilbyder', title = 'Top 5 Cummulative Revenue by Telco. Operators')
# fig.show()



import plotly.graph_objects as go

colors = ['lightslategray'] *11
colors[3] = 'crimson'

fig = go.Figure(data=[go.Bar(
    x=Yearly_share['Year'],
    y=Yearly_share['MKT_growth'],
    
    marker_color=colors 
)])
fig.update_layout(title_text='Norway Mobile Telco Market Growth')

StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1aa9059a-a88d-4b12-863b-36505d1d77d5)

## Revenue Performance (2015–2025)


- Market Dominance: Telenor and Telia dominated cumulative mobile telephony revenue, contributing 54.9% and 33.1%, respectively, representing a combined 88% market share over the period.

- Shifting Market Position: Their combined annual revenue share declined from 96.2% in 2015 to 81.1% in 2025 (−15.1 percentage points), while Lyse Tele's reported share increased from 7.5% in 2020 to 12.1% in 2025.

- Overall Revenue Growth: Total market revenue increased by 28.1% between 2015 and 2025, despite fluctuations in annual performance.

- Market Volatility: Revenue contracted by 4.8% in 2018, while growth slowed to just 0.9% in 2023.

- Recent Growth Momentum: Revenue growth accelerated to 4.6% in 2024 and 4.9% in 2025, indicating stronger recent market performance.



In [8]:
#Q4 What is the subscription volumes by operator?

subscrption_share = ekom_11.query(
    " Hovedgruppe == 'Abonnement' and Half_FullYear == 'Full Year' "
).groupby(
   
        ['Year', 'Tilbyder', 'Delkategori' ], as_index= 0
).agg(
    Total_Subscription = ('svar', 'sum')

).sort_values(
    ['Total_Subscription'], ascending = False )  


subscrption_share['Pct_of_Total'] = round(subscrption_share['Total_Subscription']/subscrption_share.groupby(['Year'])['Total_Subscription'].transform('sum') * 100,1)


#display(subscrption_share)
subscrption_share.reset_index(drop = True)


pivot_subscrption_share = subscrption_share.pivot_table(
    index = 'Tilbyder',
    columns = ['Year'],
    values = 'Total_Subscription',
    aggfunc = 'sum',
    fill_value = 0,

)
pivot_sub = pivot_subscrption_share.sort_values(
    by=pivot_subscrption_share.columns[-1],
    ascending=False
)
pivot_sub

## by pct_chnage

subscrption_share.head(60)


pivot_subscrption_pct = subscrption_share.pivot_table(
    index = 'Tilbyder',
    columns = ['Year'],
    values = 'Pct_of_Total',
    aggfunc = 'sum',
    fill_value = 0,

)
pivot_sub_pct = pivot_subscrption_pct.sort_values(
    by=pivot_subscrption_pct.columns[-1],
    ascending=False
)
pivot_sub_pct

pivot_sub.head(10)

StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 10, Finished, Available, Finished, False)

Year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Tilbyder,,,,,,,,,,,
Telenor Norge,2946175,2885913,2823612,2798209,2745031,2691497,2636670,2588474,2561819,2520956,2475133
Telia Norge,2400469,2284786,2182350,2129994,2084777,2048542,2065329,2084113,2065166,2054756,2009230
Lyse Tele,0,0,0,0,0,639390,693624,757305,882075,978504,1048751
Unifon,1993,1145,1348,13675,30471,47098,69184,114973,128029,125474,129826
Fjordkraft Mobil,7111,8218,54087,85187,122121,167429,152210,137085,114775,111958,112213
Chili Mobil,75656,84054,83756,60737,65924,70650,80380,84821,82829,88116,91955
Xplora Mobile,5750,7122,14335,19790,35466,55168,74759,82313,86061,88777,86587
Happybytes,0,0,0,1300,7562,13997,28083,41738,49440,41463,76030
Plussmobil,0,0,0,0,0,0,0,11694,45555,48518,48637


In [9]:
pivot_sub_pct.head(10)

StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 11, Finished, Available, Finished, False)

Year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Tilbyder,,,,,,,,,,,
Telenor Norge,51.6,50.4,49.4,48.9,47.5,46.1,44.5,43.0,42.3,41.3,40.4
Telia Norge,42.0,39.9,38.2,37.2,36.1,35.1,34.8,34.6,34.1,33.7,32.8
Lyse Tele,0.0,0.0,0.0,0.0,0.0,11.0,11.7,12.6,14.6,16.0,17.1
Unifon,0.0,0.0,0.0,0.2,0.5,0.8,1.2,1.9,2.1,2.1,2.1
Fjordkraft Mobil,0.1,0.1,0.9,1.5,2.1,2.9,2.6,2.3,1.9,1.8,1.8
Chili Mobil,1.3,1.5,1.5,1.1,1.1,1.2,1.4,1.4,1.4,1.4,1.5
Xplora Mobile,0.1,0.1,0.3,0.3,0.6,0.9,1.3,1.4,1.4,1.5,1.4
Happybytes,0.0,0.0,0.0,0.0,0.1,0.2,0.5,0.7,0.8,0.7,1.2
Plussmobil,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.8,0.8,0.8


### Subscription


Between 2015 and 2025, Telenor and Telia remained the dominant mobile operators, although their combined subscription market share declined from 93.6% to 73.2%, a decrease of 20.4 percentage points.
Meanwhile, Lyse Tele emerged as the third-largest reported operator, reaching 17.1% market share in 2025, although historical reporting changes involving ice must be considered.
Smaller operators, particularly Unifon (2.1%) and Fjordkraft Mobil (1.8%), also expanded their market presence, indicating a gradual shift in the distribution of mobile subscriptions beyond the two traditional market leaders

In [10]:
#Q5 How is the subscription growing?


subscrption_growth = ekom_11.query(
    " Hovedgruppe == 'Abonnement' and Half_FullYear == 'Full Year' "
).groupby(
   
        ['Year'], as_index= 0
).agg(
    Subscription_growth = ('svar', 'sum') 

).sort_values(
    ['Year'], ascending = True )  





subscription_Yearly_share = ekom_11.query(
    " Hovedgruppe == 'Abonnement' and Half_FullYear == 'Full Year' "
).groupby(
   
        ['Year'], as_index= 0
).agg(
    Subscription_growth = ('svar', 'sum') 

).sort_values(
    ['Subscription_growth'], ascending = True ).round(1)  





subscription_Yearly_share = subscription_Yearly_share.sort_values([ 'Year'])

subscription_Yearly_share['Previous_Year_sub_growth'] = (
     subscription_Yearly_share['Subscription_growth'].shift(1)
 )

subscription_Yearly_share = subscription_Yearly_share.assign(
  Subsciption_vs_LY = subscription_Yearly_share['Subscription_growth'] - subscription_Yearly_share['Previous_Year_sub_growth'],
 Yearly_growth = round(((subscription_Yearly_share['Subscription_growth']/subscription_Yearly_share['Previous_Year_sub_growth']) - 1) * 100, 1)
)
display(subscription_Yearly_share)



colors = ['lightslategray',] *11
colors[2] = 'crimson'

fig = go.Figure(data=[go.Bar(
    x=subscription_Yearly_share['Year'],
    y=subscription_Yearly_share['Yearly_growth'],
    marker_color=colors 
)])
fig.update_layout(title_text='Norway Mobile Telco MKT Growth')


StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1f14eb18-e746-49e2-96c5-5e4a70f9a4a6)

###### Summary

- Overall Growth: Total subscriptions increased from 5.71 million in 2015 to 6.12 million in 2025, representing approximately 7.1% growth over the period.

- Peak Growth: The strongest annual growth occurred in 2021 and 2022, with subscription growth of 1.6% in each year.

- Market Stability: Subscription growth remained relatively modest throughout the period, generally staying below 2% annually.

- Recent Slowdown: Annual growth declined from 0.8% in 2024 to 0.4% in 2025, indicating slower subscription expansion.

In [11]:
#Q6 Which operators outperform the market?

Market_share = ekom_11.query(
    " Hovedgruppe == 'Inntekter'"
).groupby(
   
        ['Year','Tilbyder'], as_index= 0
).agg(
    Total_Revenue = ('svar', 'sum')

).sort_values(['Total_Revenue'], ascending = False )



Market_share = Market_share.sort_values(['Tilbyder', 'Year'])

Market_share['Previous_Year_Revenue'] = (
    Market_share.groupby('Tilbyder')['Total_Revenue'].shift(1)
)

Market_share['Operators_growth'] = round(((Market_share['Total_Revenue']/Market_share['Previous_Year_Revenue']) - 1) * 100, 1)


Market_share.reset_index(drop = True)



## Merging with the market share table


Market_performers = Market_share.merge(
    Yearly_share[['Year','MKT_growth']],
    how = 'left',
    left_on = 'Year',
    right_on = 'Year'
)

Market_performers = Market_performers.assign(
    Growth_diff = Market_performers['Operators_growth'] - Market_performers['MKT_growth'],
    Growth_bracket = lambda perf : np.where(perf['Growth_diff'] < 0, 'Under performing',
                    np.where(perf['Growth_diff']  == 0, 'Performance Par',
                    np.where(perf['Growth_diff'] > 0, 'Over performing', 'None'
                    
                    ))))






overperformers = Market_performers.query(
    "Growth_bracket == 'Over performing'"
).sort_values(['Year','Growth_diff'], ascending = [1,0])


display(overperformers)


StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e1fe2f27-be93-4a82-b035-af981e993a32)

####Summary:

Between 2016 and 2025, Xplora Mobile demonstrated the most consistent relative revenue growth, outperforming the Norwegian mobile telephony market in all 10 years.

Unifon and Chili Mobil also recorded recurring outperformance, exceeding market growth in seven years each, while Happybytes outperformed in six years.

Lyse Tele demonstrated strong recent momentum, outperforming the market in every year from 2021 to 2025. Its relatively large revenue base makes this performance particularly significant compared with smaller operators.

In [12]:
#Q7 Is growth stronger in consumer or business markets?

df_rev = ekom_11.query(
    " Hovedgruppe == 'Inntekter'"
).groupby(
   
        ['Year','Tilbyder', 'NivÃ¥1', 'Markedssegment'], as_index= 0
).agg(
    Total_Revenue = ('svar', 'sum')

).sort_values(['Year'], ascending = True ).reset_index(drop = True)

df_rev = df_rev.assign(
  Per_Year_Total_Revenue =  df_rev.groupby(['Year'])['Total_Revenue'].transform('sum'),
  Pct_of_total_Rev = lambda pct : ((pct['Total_Revenue']/pct['Per_Year_Total_Revenue']) * 100).round(1),
  Previous_Year_Revenue = lambda LY: LY.groupby(['Tilbyder','NivÃ¥1', 'Markedssegment'], as_index = 0)['Total_Revenue'].shift(1),
  Pct_of_Rev_Growth =  lambda pct_ly : ((pct_ly['Total_Revenue']/pct_ly['Previous_Year_Revenue']-1) * 100).round(1),
  Revenue_diff =  lambda pct_ly : (pct_ly['Total_Revenue']-pct_ly['Previous_Year_Revenue']).round(1)
  ).fillna(0)



df_abonnement = ekom_11.query(
    " Hovedgruppe == 'Abonnement' "
).groupby(
   
        ['Year','Tilbyder', 'NivÃ¥1', 'Markedssegment'], as_index= 0
).agg(
    Total_Abonnement = ('svar', 'sum')

).sort_values(['Year'], ascending = True ).reset_index(drop = True)

df_abonnement = df_abonnement.assign(
  Per_Year_Total_Abonnement =  df_abonnement.groupby(['Year'])['Total_Abonnement'].transform('sum'),
  Pct_of_total_Abonnement = lambda pct : ((pct['Total_Abonnement']/pct['Per_Year_Total_Abonnement']) * 100).round(1),
  Previous_Year_Abonnement = lambda LY: LY.groupby(['Tilbyder','NivÃ¥1', 'Markedssegment'], as_index = 0)['Total_Abonnement'].shift(1),
  Pct_of_Abonnement_Growth =  lambda pct_ly : ((pct_ly['Total_Abonnement']/pct_ly['Previous_Year_Abonnement']-1) * 100).round(1),
 Abonnement_diff =  lambda pct_ly : (pct_ly['Total_Abonnement']-pct_ly['Previous_Year_Abonnement']).round(1)
  ).fillna(0)



display(df_rev)
#whrn you start with the pct_change, don't use the Year


StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3bd08006-5581-4873-9c84-de22e2b02176)

In [13]:


markedgrowth = df_rev.groupby(['Year', 'Tilbyder','Markedssegment'], as_index = 0).agg(
    Total_Revenue = ('Total_Revenue', 'sum'),
    Pct_Rev = ('Pct_of_total_Rev', 'sum')
).sort_values(['Year'], ascending = 0)

display(markedgrowth)


markedgrowth_pct = df_rev.groupby(['Markedssegment'], as_index = 0).agg(
    Total_Revenue = ('Total_Revenue', 'sum')
).sort_values(['Total_Revenue'], ascending = 0)

markedgrowth_pct = markedgrowth_pct.assign(
      MKT_pct = lambda mkt_pct:( mkt_pct['Total_Revenue']/mkt_pct['Total_Revenue'].sum() * 100).round(1)
)


display(markedgrowth_pct)






StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37aa86d7-392f-425f-b5e3-d7ff57a2a268)

SynapseWidget(Synapse.DataFrame, f6724227-5929-4a8d-bd0e-dcd9ca9bd735)

### Summary

Between 2015 and 2025, the Norwegian mobile telephony consumer segment (Privat) experienced stronger revenue growth of 70.7% than the business segment (Bedrift) of 29.3%

Consumer revenue increased from 10.95 million in 2015 to 15.42 million in 2025, representing approximately 40.8% growth. In contrast, business revenue increased marginally from 5.51 million to 5.68 million, representing approximately 3.0% growth.

While consumer revenue generally increased throughout the period, business revenue remained relatively stable, with some fluctuations.

In [14]:
#Q.8 Which mobile operators are most dependent on the slower-growing business segment?




#display(declining_growth)
bedrift = df_rev.query("Markedssegment == 'Bedrift'").groupby(['Year', 'Tilbyder', 'Markedssegment'], as_index = 0).agg(
    Bedrift_Revenue = ('Total_Revenue', 'sum')
)


bedrift



all_rev = markedgrowth.merge(
    bedrift,
    how = 'left',
    left_on = ['Year', 'Tilbyder', 'Markedssegment'],
    right_on = ['Year', 'Tilbyder', 'Markedssegment']
)


all_revs = all_rev.drop(['Pct_Rev', 'Markedssegment'], axis = 1)
all_revs = all_revs.assign(
    Segment_Revenue = all_revs.groupby(['Year', 'Tilbyder'])['Total_Revenue'].transform('sum'),
    Pct_of_bedrift_Reve = lambda bedrift : ( bedrift['Bedrift_Revenue']/bedrift['Segment_Revenue'] * 100).round(1) )




df = all_revs[all_revs['Pct_of_bedrift_Reve'].notna()].drop(['Total_Revenue'], axis = 1)
display(df)

StatementMeta(, f0aee44c-b7be-408d-92ba-aec9dfd3de2e, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e678964c-792d-490b-9135-6f16abeb1fc1)

##### Key Findings

Analysis of Norwegian mobile telephony revenue between 2015 and 2025 reveals substantial differences in operators' dependence on the business segment.

Unifon consistently generated nearly all its reported revenue from business customers, maintaining approximately 99% business revenue exposure in 2025. Similarly, Smb Mobil recorded 98.3% exposure in 2025, indicating a highly business-focused revenue structure.

Among the larger operators, Telenor's business revenue exposure declined from 38.5% in 2015 to 30.8% in 2025, while Telia's remained relatively stable, increasing slightly from 24.1% to 25.0%.

In contrast, Lyse Tele and Happybytes were predominantly consumer-focused, with business revenue accounting for only 9.7% and 1.7% of their respective revenues in 2025.